## Comparación de modelos: Framingham

In [1]:
import mlflow

from src.data import load_dataset
from src.models import build_lightgbm, build_tabpfn
from src.experiments import run_experiment
from src.optimization import optimize_experiment

participant = "TU_NOMBRE"
apikey = "tabpfn_sk_XnevfzVjqKQK1bOy0ExbAoxguuZGdQJD3dYOHEHlLWE"
mlflow.set_experiment("experimentos")


<Experiment: artifact_location='file:c:/Users/usuario/root/usc/sdg/Modelos_Fundacionales/mlruns/4', creation_time=1793261501280, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1793261501280, lifecycle_stage='active', name='experimentos', tags={}, trace_location=None, workspace='default'>

In [2]:
dataset = "framingham"  

X_train, X_test, y_train, y_test = load_dataset(
    dataset=dataset,
    random_state=42,
    test_size=0.2,
    one_hot=False,
)

print(f"Dataset: {dataset}")
print(f"Entrenamiento: {X_train.shape} | Test: {X_test.shape}")


Dataset: framingham
Entrenamiento: (3392, 15) | Test: (848, 15)


### LightGBM


In [11]:
params_lgbm = {
    "n_estimators": 200,
    "learning_rate": 0.01,
    "random_state": 42,
    "verbosity": -1,
}

model_lgbm, metrics_lgbm = run_experiment(
    model=build_lightgbm(params_lgbm),
    model_name="LightGBM",
    dataset=dataset,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    params=params_lgbm,
    participant=participant,
)

print(metrics_lgbm)


{'recall': 0.05426356589147287, 'f1': 0.10071942446043165, 'roc_auc': 0.6558905025282745, 'pr_auc': 0.27502901158377835, 'balanced_accuracy': 0.5250455520695195, 'precision': 0.7}


### TabPFN

In [12]:
params_tabpfn = {}

model_tabpfn, metrics_tabpfn = run_experiment(
    model=build_tabpfn(params_tabpfn, apikey=apikey),
    model_name="TabPFN",
    dataset=dataset,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    params=params_tabpfn,
    participant=participant,
)

print(metrics_tabpfn)


{'recall': 0.046511627906976744, 'f1': 0.08450704225352113, 'roc_auc': 0.701491089044862, 'pr_auc': 0.29481842076562204, 'balanced_accuracy': 0.5183879419089821, 'precision': 0.46153846153846156}


### Optimización de LightGBM

In [3]:
X_train_raw, X_test_raw, y_train_opt, y_test_opt = load_dataset(
    dataset=dataset, preprocess=False, random_state=42, test_size=0.2
)

optimization_result = optimize_experiment(
    dataset=dataset,
    model=build_lightgbm({"random_state": 42, "verbosity": -1, "n_jobs": 2}),
    model_name="LightGBM_Opimization",
    X_train=X_train_raw, X_test=X_test_raw,
    y_train=y_train_opt, y_test=y_test_opt,
    param_distributions={
        "n_estimators": [100, 200, 300],
        "learning_rate": [0.01, 0.05, 0.1],
        "num_leaves": [15, 31],
    },
    n_iter=5,
    cv_folds=3,
    random_state=42,
    one_hot=True,
    min_precision=0.30,  # None para no optimizar el umbral
    participant=participant,
)

print("Mejores parámetros:", optimization_result["best_params"])
print("Umbral:", optimization_result["threshold"])
print("Test con umbral 0.5:", optimization_result["test_default"])
print("Test con umbral optimizado:", optimization_result["test_tuned"])


Mejores parámetros: {'num_leaves': 15, 'n_estimators': 100, 'learning_rate': 0.01}
Umbral: {'threshold': 0.19168081563262207, 'recall_cv': 0.4368932038834951, 'precision_cv': 0.3}
Test con umbral 0.5: {'recall': 0.0, 'f1': 0.0, 'roc_auc': 0.6704240385548403, 'pr_auc': 0.2818065588340789, 'balanced_accuracy': 0.5, 'precision': 0.0}
Test con umbral optimizado: {'recall': 0.40310077519379844, 'f1': 0.33121019108280253, 'roc_auc': 0.6704240385548403, 'pr_auc': 0.2818065588340789, 'balanced_accuracy': 0.6090608187512803, 'precision': 0.2810810810810811}
